# EXP-02 — Line-1 Protected / Confidence-Gated Retraction

## 1 — Experiment definition

| | EXP-01 | EXP-02 |
|---|---|---|
| **Name** | Right-Most Town / All-Match Country Retraction Baseline | Line-1 Protected / Confidence-Gated Retraction |
| **Slug** | `exp01_rightmost_town_allmatch_country` | `exp02_line1_protected_confidence_gated` |
| **Town** | one right-most verified occurrence removed per group | the same, but only among **eligible** fields and only if `predicted_town_probability` **> 0.80** |
| **Country** | all verified occurrences/forms removed | the same, but only among **eligible** fields and only if `predicted_country_probability` **> 0.80** |
| **Protected fields** | none | the first configured source field (`address_line_1`) is never modified |
| **Everything else** | — | unchanged: prompt, model, input, extraction, verification, reference data, probabilities, Composite Weighted Score, cross-entropy, HITL |

EXP-02 changes **retraction eligibility only**. The gates are strict (`>` not `>=`), so a
probability of exactly 0.80 does not retract, and Town and Country are gated independently.
Both runs read the same 115 input records, the same cache of model responses, and the same
scoring; any difference in a prediction field between them is contamination, not a result,
and section 3 checks for it before anything is compared.

> This notebook measures policy behaviour. It reads what the two completed runs wrote, makes no
> model call, re-runs nothing, and does not establish semantic correctness without labelled
> retraction ground truth.

## 2 — Load both experiments

Paths come from each experiment's own configuration via `load_config`, never from guesses. The
`EXPERIMENTS` registry and the helpers beneath it are what a future EXP-03 plugs into (section
11). Missing artifacts are reported, not fatal.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd


def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pytest.ini").is_file() and (candidate / "models").is_dir():
            return candidate
    raise RuntimeError(f"Could not locate the repository root from {start}.")


REPO_ROOT = _find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from models.swft_tc.src.settings import MODEL_ROOT, load_config

GROUP = "1"
OUTPUT_ROOT = MODEL_ROOT / "outputs"
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 240)

# --- the experiment registry: add EXP-03 here and the rest of the notebook follows ----------
EXPERIMENTS = {
    "EXP-01": {
        "name": "Right-Most Town / All-Match Country Retraction Baseline",
        "config": MODEL_ROOT / "config" / "config_exp01_rightmost_town_allmatch_country.yaml",
    },
    "EXP-02": {
        "name": "Line-1 Protected / Confidence-Gated Retraction",
        "config": MODEL_ROOT / "config" / "config_exp02_line1_protected_confidence_gated.yaml",
    },
}
BASELINE_ID, CHALLENGER_ID = "EXP-01", "EXP-02"

for exp_id, spec in EXPERIMENTS.items():
    cfg = load_config(spec["config"], base_dir=MODEL_ROOT)
    spec["cfg"] = cfg
    spec["output"] = cfg.path(cfg.processing.output_path)
    spec["detailed"] = cfg.path(cfg.processing.detailed_json_path)
    spec["metrics"] = cfg.path(cfg.processing.metrics_path)
    spec["policy"] = cfg.retraction

config = EXPERIMENTS[CHALLENGER_ID]["cfg"]      # column templates are shared by both
RECORD_ID = config.project.record_id_column
SOURCE_FIELDS = ["address_line_1", "address_line_2", "address_line_3", "address_line_4"]


def col(key: str) -> str:
    return config.output.column_name(key, GROUP)


C = {k: col(k) for k in (
    "combined_address_cleaned", "predicted_town", "predicted_country", "predicted_country_name",
    "predicted_town_probability", "predicted_country_probability", "predicted_town_exists",
    "predicted_country_exists", "composite_weighted_score", "combined_address_retracted",
    "combined_address_retracted_comments", "hitl_flag", "hitl_state", "hitl_state_reason",
)}
EXTRACTION_FIELDS = [C[k] for k in (
    "combined_address_cleaned", "predicted_town", "predicted_country",
    "predicted_town_probability", "predicted_country_probability",
    "predicted_town_exists", "predicted_country_exists",
)]


def _rel(path: Path) -> str:
    try:
        return str(Path(path).relative_to(REPO_ROOT))
    except ValueError:
        return str(path)


print("repository root :", REPO_ROOT)
print("model root      :", MODEL_ROOT.relative_to(REPO_ROOT))
for exp_id, spec in EXPERIMENTS.items():
    pol = spec["policy"]
    print(f"{exp_id}: {spec['name']}")
    print(f"   policy {pol.policy_name!r} protected={list(pol.protected_source_positions)} "
          f"town>{pol.town_probability_threshold} country>{pol.country_probability_threshold}")
    print(f"   output   {_rel(spec['output'])}")

In [ ]:
NOT_READY = ("Experiment output is not available yet. Complete the batch run "
             "(see README → Retraction Experiments) and rerun this cell.")


def as_bool(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().str.lower().eq("true")


def as_num(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")


def pct(count: int, total: int) -> str:
    return f"{count} ({100.0 * count / total:.1f}%)" if total else "0"


def show(frame: pd.DataFrame, n: int = 10) -> None:
    print(f"{len(frame)} row(s){' — showing ' + str(n) if len(frame) > n else ''}")
    if len(frame):
        display(frame.head(n))


def load_experiment(exp_id: str) -> dict | None:
    """Load one experiment's canonical CSV, detailed audit and metrics, or None if absent."""
    spec = EXPERIMENTS[exp_id]
    missing = [k for k in ("output", "detailed") if not spec[k].exists()]
    if missing:
        print(f"{exp_id}: missing {missing} -> {NOT_READY}")
        return None
    frame = pd.read_csv(spec["output"], dtype=str, keep_default_na=False, encoding="utf-8-sig")
    with spec["detailed"].open(encoding="utf-8") as fh:
        detail = [json.loads(line) for line in fh if line.strip()]
    metrics = json.loads(spec["metrics"].read_text(encoding="utf-8")) if spec["metrics"].exists() else {}
    audit = pd.DataFrame([
        {RECORD_ID: str(rec["record_id"]), **{
            f"retr_{k}": v for k, v in rec["groups"][GROUP].get("retraction", {}).items()
            if k not in ("actual_column_before_retraction", "actual_column_after_retraction")
        },
         "line1_before": rec["groups"][GROUP].get("retraction", {}).get("actual_column_before_retraction", {}).get(SOURCE_FIELDS[0]),
         "line1_after": rec["groups"][GROUP].get("retraction", {}).get("actual_column_after_retraction", {}).get(SOURCE_FIELDS[0]),
         "scenario": rec["groups"][GROUP].get("scoring", {}).get("scenario"),
        }
        for rec in detail if GROUP in rec.get("groups", {})
    ])
    print(f"{exp_id}: {len(frame)} rows, {len(detail)} detailed records, "
          f"run {metrics.get('run', {}).get('started_at_utc', '?')}, "
          f"backend calls {metrics.get('efficiency', {}).get('backend_calls', '?')}")
    return {"frame": frame, "detail": detail, "metrics": metrics, "audit": audit}


RUNS = {exp_id: load_experiment(exp_id) for exp_id in EXPERIMENTS}
BOTH_READY = all(RUNS[k] is not None for k in (BASELINE_ID, CHALLENGER_ID))


def ready(*ids: str) -> bool:
    ids = ids or (BASELINE_ID, CHALLENGER_ID)
    ok = all(RUNS.get(i) is not None for i in ids)
    if not ok:
        print(NOT_READY)
    return ok

## 3 — Experiment integrity

A controlled comparison is only valid if the two runs share every extraction-side field. The
check is by `RECORD_ID` over the original four lines, the cleaned combined address, both
predictions, both probabilities and both `*_exists` flags. `experiment_contamination_flag` must
be `False` for every record; if it is not, the differing fields are shown and the retraction
comparison below is **not** to be trusted.

In [ ]:
def validate_experiment_identity(baseline: pd.DataFrame, challenger: pd.DataFrame,
                                 fields: list[str]) -> pd.DataFrame:
    """Per-record contamination report over the extraction-side fields."""
    a = baseline.set_index(RECORD_ID)[fields]
    b = challenger.set_index(RECORD_ID)[fields]
    ids = sorted(set(a.index) | set(b.index))
    rows = []
    for rid in ids:
        if rid not in a.index or rid not in b.index:
            rows.append({RECORD_ID: rid, "experiment_contamination_flag": True,
                         "differing_fields": "record missing from one experiment"})
            continue
        diff = [f for f in fields if str(a.at[rid, f]) != str(b.at[rid, f])]
        rows.append({RECORD_ID: rid, "experiment_contamination_flag": bool(diff),
                     "differing_fields": ", ".join(diff)})
    return pd.DataFrame(rows)


if ready():
    identity = validate_experiment_identity(
        RUNS[BASELINE_ID]["frame"], RUNS[CHALLENGER_ID]["frame"], [*SOURCE_FIELDS, *EXTRACTION_FIELDS])
    contaminated = identity.loc[identity["experiment_contamination_flag"]]
    CONTAMINATED = not contaminated.empty
    print(f"records compared: {len(identity)} | contaminated: {len(contaminated)}")
    if CONTAMINATED:
        print("\n!!! EXPERIMENT CONTAMINATION — extraction-side fields differ. "
              "The retraction comparison below is NOT a controlled comparison. !!!")
        show(contaminated, n=50)
    else:
        print("experiment_contamination_flag is False for every record: the comparison is controlled.")

## 4 — Side-by-side master table

One row per `RECORD_ID`, shared fields once, experiment-specific fields prefixed `EXP01_` /
`EXP02_`. The reason columns come from the challenger's audit block (`*_skip_reason`,
`protected_source_fields`), not from text heuristics.

In [ ]:
def compare_experiments(baseline: dict, challenger: dict) -> pd.DataFrame:
    """Join the two runs by RECORD_ID with prefixed experiment columns and change reasons."""
    shared = [RECORD_ID, *SOURCE_FIELDS, C["predicted_town"], C["predicted_country"],
              C["predicted_town_probability"], C["predicted_country_probability"],
              C["predicted_town_exists"], C["predicted_country_exists"], C["combined_address_cleaned"]]
    a = baseline["frame"]; b = challenger["frame"]
    m = a[shared].rename(columns={
        C["predicted_town"]: "predicted_town", C["predicted_country"]: "predicted_country",
        C["predicted_town_probability"]: "town_probability",
        C["predicted_country_probability"]: "country_probability",
        C["predicted_town_exists"]: "town_exists", C["predicted_country_exists"]: "country_exists",
        C["combined_address_cleaned"]: "combined_address_cleaned"})
    for tag, run in (("EXP01", baseline), ("EXP02", challenger)):
        f = run["frame"][[RECORD_ID, C["combined_address_retracted"], C["combined_address_retracted_comments"],
                          C["hitl_state"], C["composite_weighted_score"]]].rename(columns={
            C["combined_address_retracted"]: f"{tag}_retracted_address",
            C["combined_address_retracted_comments"]: f"{tag}_comment",
            C["hitl_state"]: f"{tag}_HITL_state", C["composite_weighted_score"]: f"{tag}_score"})
        m = m.merge(f, on=RECORD_ID, how="outer")
        au = run["audit"][[RECORD_ID, "retr_retracted_entities", "retr_town_skip_reason",
                           "retr_country_skip_reason", "retr_town_probability_gate_passed",
                           "retr_country_probability_gate_passed", "line1_before", "line1_after"]].rename(
            columns=lambda c: c if c == RECORD_ID else f"{tag}_{c.removeprefix('retr_')}")
        m = m.merge(au, on=RECORD_ID, how="left")

    m["town_probability"] = as_num(m["town_probability"]); m["country_probability"] = as_num(m["country_probability"])
    m["exp01_vs_exp02_changed"] = m["EXP01_retracted_address"].ne(m["EXP02_retracted_address"])
    m["EXP01_line1_changed"] = m["EXP01_line1_before"].ne(m["EXP01_line1_after"])
    m["EXP02_line1_changed"] = m["EXP02_line1_before"].ne(m["EXP02_line1_after"])
    m["line1_protection_changed_result"] = m["exp01_vs_exp02_changed"] & m["EXP01_line1_changed"]
    m["town_probability_gate_changed_result"] = (
        m["exp01_vs_exp02_changed"] & m["EXP02_town_skip_reason"].eq("probability_not_above_threshold")
        & m["EXP01_retracted_entities"].astype(str).str.contains("town"))
    m["country_probability_gate_changed_result"] = (
        m["exp01_vs_exp02_changed"] & m["EXP02_country_skip_reason"].eq("probability_not_above_threshold")
        & m["EXP01_retracted_entities"].astype(str).str.contains("country"))

    def reason(row) -> str:
        if not row["exp01_vs_exp02_changed"]:
            return ""
        parts = []
        if row["line1_protection_changed_result"]:
            parts.append("line1_protected")
        if row["town_probability_gate_changed_result"]:
            parts.append("town_probability_gate")
        if row["country_probability_gate_changed_result"]:
            parts.append("country_probability_gate")
        if row["EXP02_town_skip_reason"] == "protected_source_only":
            parts.append("town_only_in_protected_line")
        if row["EXP02_country_skip_reason"] == "protected_source_only":
            parts.append("country_only_in_protected_line")
        return ",".join(dict.fromkeys(parts)) or "other_retraction_difference"

    m["comparison_reason"] = m.apply(reason, axis=1)
    return m


if ready() and not CONTAMINATED:
    master = compare_experiments(RUNS[BASELINE_ID], RUNS[CHALLENGER_ID])
    MASTER_VIEW = [RECORD_ID, *SOURCE_FIELDS, "predicted_town", "predicted_country", "town_probability",
                   "country_probability", "EXP01_retracted_address", "EXP02_retracted_address",
                   "EXP01_HITL_state", "EXP02_HITL_state", "exp01_vs_exp02_changed",
                   "line1_protection_changed_result", "town_probability_gate_changed_result",
                   "country_probability_gate_changed_result", "comparison_reason"]
    print(f"{len(master)} records; {int(master['exp01_vs_exp02_changed'].sum())} with a different retracted address")
    show(master[MASTER_VIEW], n=10)

## 5 — Executive comparison

Policy metrics side by side. **HITL and auto-accept counts are expected to be identical** —
EXP-02 did not touch scoring or routing — so a non-zero delta there is unexpected and is flagged.

In [ ]:
from models.swft_tc.src.retraction import token_phrase_matches

ORGANISATION_TOKENS = ("BANK", "BANQUE", "BANCO", "CITIBANK", "CORP", "CORPORATION", "INC", "LIMITED",
                       "LTD", "PLC", "BRANCH", "OFFICE", "TRUST", "FINANCIAL", "SECURITIES", "CAPITAL",
                       "GROUP", "AUTHORITY", "COMPANY", "SA", "SAE", "LLC", "NA")


def entity_collision_candidate(frame: pd.DataFrame) -> pd.Series:
    """Notebook-only manual-review heuristic (same as the EXP-01 notebook). Not a correctness label."""
    cleaned = frame[C["combined_address_cleaned"]].astype(str)
    has_org = cleaned.map(lambda s: any(token_phrase_matches(s, tok) for tok in ORGANISATION_TOKENS))
    verified = as_bool(frame[C["predicted_town_exists"]]) | as_bool(frame[C["predicted_country_exists"]])
    changed = cleaned.ne(frame[C["combined_address_retracted"]].astype(str))
    return has_org & verified & changed


def summarize_retraction_metrics(run: dict) -> dict:
    f = run["frame"]
    a = run["audit"].set_index(RECORD_ID).reindex(f[RECORD_ID]).reset_index()
    ents = a["retr_retracted_entities"].astype(str)
    cleaned, retracted = f[C["combined_address_cleaned"]], f[C["combined_address_retracted"]]
    return {
        "Total records": len(f),
        "Any retraction": int(cleaned.ne(retracted).sum()),
        "No retraction": int(cleaned.eq(retracted).sum()),
        "Line-1 changed": int(a["line1_before"].ne(a["line1_after"]).sum()),
        "Town retracted": int(ents.str.contains("town").sum()),
        "Country retracted": int(ents.str.contains("country").sum()),
        "Both retracted": int((ents.str.contains("town") & ents.str.contains("country")).sum()),
        "Town not retracted due probability": int(a["retr_town_skip_reason"].eq("probability_not_above_threshold").sum()),
        "Country not retracted due probability": int(a["retr_country_skip_reason"].eq("probability_not_above_threshold").sum()),
        "Protected-line-only Town": int(a["retr_town_skip_reason"].eq("protected_source_only").sum()),
        "Protected-line-only Country": int(a["retr_country_skip_reason"].eq("protected_source_only").sum()),
        "HITL": int(as_bool(f[C["hitl_flag"]]).sum()),
        "AUTO_ACCEPT": int((~as_bool(f[C["hitl_flag"]])).sum()),
        "Possible entity collision candidates": int(entity_collision_candidate(f).sum()),
    }


if ready() and not CONTAMINATED:
    s1, s2 = summarize_retraction_metrics(RUNS[BASELINE_ID]), summarize_retraction_metrics(RUNS[CHALLENGER_ID])
    exec_table = pd.DataFrame({"EXP-01": s1, "EXP-02": s2})
    exec_table["Delta"] = exec_table["EXP-02"] - exec_table["EXP-01"]
    display(exec_table)
    for metric in ("HITL", "AUTO_ACCEPT"):
        if exec_table.at[metric, "Delta"] != 0:
            print(f"!!! UNEXPECTED: {metric} differs between experiments (delta {exec_table.at[metric, 'Delta']}). "
                  "EXP-02 must not change routing — investigate before trusting this comparison.")
    if exec_table.at["Line-1 changed", "EXP-02"] != 0:
        print("!!! UNEXPECTED: EXP-02 modified address_line_1 — the protection did not hold.")

## 6 — Line-1 protection

Experimental proof that EXP-02 never changes `address_line_1`: the count must be 0. The records
EXP-01 did change are the ones the protection rescued.

In [ ]:
if ready() and not CONTAMINATED:
    print("records where EXP-01 changed line 1 :", int(master["EXP01_line1_changed"].sum()))
    print("records where EXP-02 changed line 1 :", int(master["EXP02_line1_changed"].sum()), "(expected 0)")
    rescued = master.loc[master["EXP01_line1_changed"] & ~master["EXP02_line1_changed"]]
    print("\nline 1 changed by EXP-01 and preserved by EXP-02:")
    show(rescued[[RECORD_ID, "address_line_1", "EXP01_line1_after", "EXP02_line1_after",
                  "predicted_town", "predicted_country", "comparison_reason"]], n=15)

## 7 — Probability-gate analysis

Cohorts at and around the gate, Town and Country independently. Exactly 0.80 sits on the
non-retracting side.

In [ ]:
if ready() and not CONTAMINATED:
    gate = EXPERIMENTS[CHALLENGER_ID]["policy"]
    tt, ct = gate.town_probability_threshold, gate.country_probability_threshold
    tp, cp = master["town_probability"], master["country_probability"]
    cohorts = pd.DataFrame({
        "rows": [int(tp.le(tt).sum()), int(tp.gt(tt).sum()), int(tp.eq(tt).sum()),
                 int(cp.le(ct).sum()), int(cp.gt(ct).sum()), int(cp.eq(ct).sum())]},
        index=[f"Town probability <= {tt}", f"Town probability > {tt}", f"Town probability == {tt} exactly",
               f"Country probability <= {ct}", f"Country probability > {ct}", f"Country probability == {ct} exactly"])
    display(cohorts)
    print("blocked by the Town gate            :", int(master["EXP02_town_skip_reason"].eq("probability_not_above_threshold").sum()))
    print("blocked by the Country gate         :", int(master["EXP02_country_skip_reason"].eq("probability_not_above_threshold").sum()))
    print("gate changed the final retraction   :",
          int((master["town_probability_gate_changed_result"] | master["country_probability_gate_changed_result"]).sum()))
    near = master.loc[tp.between(tt - 0.05, tt + 0.05) | cp.between(ct - 0.05, ct + 0.05)]
    print("\nexamples within ±0.05 of a gate:")
    show(near[[RECORD_ID, "predicted_town", "town_probability", "predicted_country", "country_probability",
               "EXP02_town_probability_gate_passed", "EXP02_country_probability_gate_passed",
               "EXP01_retracted_address", "EXP02_retracted_address"]], n=12)

## 8 — High-value examples

The organisation-name cases the experiment was designed around, side by side.

In [ ]:
HIGH_VALUE_IDS = ["1208", "1212", "2152", "1578", "1450"]

if ready() and not CONTAMINATED:
    idx = master.set_index(RECORD_ID)
    for rid in HIGH_VALUE_IDS:
        if rid not in idx.index:
            print(f"[{rid}] not in the experiment input\n"); continue
        r = idx.loc[rid]
        print(f"[{rid}]  line 1          : {r['address_line_1']}")
        print(f"        Town/Country    : {r['predicted_town']} (p={r['town_probability']}) / "
              f"{r['predicted_country']} (p={r['country_probability']})")
        print(f"        EXP-01 retracted: {r['EXP01_retracted_address']}")
        print(f"        EXP-02 retracted: {r['EXP02_retracted_address']}")
        print(f"        changed         : {r['exp01_vs_exp02_changed']}  reason: {r['comparison_reason'] or '—'}")
        print(f"        EXP-02 comment  : {r['EXP02_comment']}\n")

## 9 — Semantic-risk review (manual prioritisation only)

The EXP-01 notebook's organisation-token heuristic, applied to both runs. It prioritises human
review; it is not a correctness label and no row is called right or wrong.

In [ ]:
if ready() and not CONTAMINATED:
    c1 = entity_collision_candidate(RUNS[BASELINE_ID]["frame"])
    c2 = entity_collision_candidate(RUNS[CHALLENGER_ID]["frame"])
    ids1 = set(RUNS[BASELINE_ID]["frame"].loc[c1, RECORD_ID]); ids2 = set(RUNS[CHALLENGER_ID]["frame"].loc[c2, RECORD_ID])
    print(f"possible entity/location collision candidates — EXP-01: {len(ids1)} | EXP-02: {len(ids2)} | "
          f"no longer flagged under EXP-02: {len(ids1 - ids2)} | newly flagged: {len(ids2 - ids1)}")
    still = master.loc[master[RECORD_ID].isin(ids2)]
    print("\nstill a review candidate under EXP-02 (retraction changed an address carrying an organisation token):")
    show(still[[RECORD_ID, "address_line_1", "predicted_town", "predicted_country",
                "EXP02_retracted_address", "comparison_reason"]], n=15)

## 10 — Retraction delta categories

Every record whose retracted address differs gets one explainable category, derived from the
audit reasons. Records can carry several reasons; the category reflects their combination.

In [ ]:
def delta_category(row) -> str:
    if not row["exp01_vs_exp02_changed"]:
        return "UNCHANGED_FROM_EXP01"
    line1 = row["line1_protection_changed_result"] or row["EXP02_town_skip_reason"] == "protected_source_only" \
        or row["EXP02_country_skip_reason"] == "protected_source_only"
    town_gate = row["town_probability_gate_changed_result"]
    country_gate = row["country_probability_gate_changed_result"]
    if line1 and (town_gate or country_gate):
        return "LINE1_AND_PROBABILITY_GATE"
    if line1:
        return "LINE1_PROTECTION_ONLY"
    if town_gate and not country_gate:
        return "TOWN_PROBABILITY_GATE_ONLY"
    if country_gate and not town_gate:
        return "COUNTRY_PROBABILITY_GATE_ONLY"
    if town_gate and country_gate:
        return "TOWN_AND_COUNTRY_PROBABILITY_GATE"
    return "OTHER_RETRACTION_DIFFERENCE"


if ready() and not CONTAMINATED:
    master["delta_category"] = master.apply(delta_category, axis=1)
    display(master["delta_category"].value_counts().rename_axis("category").to_frame("rows"))
    other = master.loc[master["delta_category"].eq("OTHER_RETRACTION_DIFFERENCE")]
    if len(other):
        print("\nOTHER_RETRACTION_DIFFERENCE — differences the audit reasons do not explain; inspect:")
        show(other[[RECORD_ID, "address_line_1", "EXP01_retracted_address", "EXP02_retracted_address",
                    "EXP01_comment", "EXP02_comment"]], n=10)

## 11 — Future-experiment framework

Everything above is driven by `EXPERIMENTS`, `load_experiment`, `validate_experiment_identity`,
`compare_experiments`, `summarize_retraction_metrics` and `delta_category`. To add EXP-03:
register its config in `EXPERIMENTS`, set `CHALLENGER_ID`, and rerun. The cell below shows the
pattern on every registered pair against the baseline, so new experiments appear automatically.

In [ ]:
if ready(BASELINE_ID):
    for exp_id in [k for k in EXPERIMENTS if k != BASELINE_ID]:
        if RUNS.get(exp_id) is None:
            print(f"{exp_id}: not available yet"); continue
        ident = validate_experiment_identity(RUNS[BASELINE_ID]["frame"], RUNS[exp_id]["frame"],
                                             [*SOURCE_FIELDS, *EXTRACTION_FIELDS])
        contaminated_n = int(ident["experiment_contamination_flag"].sum())
        cmp = compare_experiments(RUNS[BASELINE_ID], RUNS[exp_id])
        print(f"{BASELINE_ID} vs {exp_id}: contaminated={contaminated_n}, "
              f"retracted address differs in {int(cmp['exp01_vs_exp02_changed'].sum())} / {len(cmp)} records")

## 12 — Conclusions

Generated from the values above. These are statements about **policy behaviour** on this
input; they do not establish semantic correctness without labelled retraction ground truth.

In [ ]:
if ready() and not CONTAMINATED:
    n = len(master)
    rescued_n = int((master["EXP01_line1_changed"] & ~master["EXP02_line1_changed"]).sum())
    town_sup = int(master["town_probability_gate_changed_result"].sum())
    country_sup = int(master["country_probability_gate_changed_result"].sum())
    changed_n = int(master["exp01_vs_exp02_changed"].sum())
    hitl_delta = int(exec_table.at["HITL", "Delta"])
    print(f"- Experiment integrity: 0 of {n} records contaminated; the comparison is controlled.")
    print(f"- EXP-02 preserved address_line_1 in {rescued_n} additional records relative to EXP-01, "
          f"and modified it in {int(master['EXP02_line1_changed'].sum())} records.")
    print(f"- {town_sup} Town retractions and {country_sup} Country retractions were suppressed by the "
          f"> {EXPERIMENTS[CHALLENGER_ID]['policy'].town_probability_threshold} probability gate.")
    print(f"- The retracted address differs in {changed_n} / {n} records; categories are in section 10.")
    print(f"- HITL routing delta: {hitl_delta} (expected 0; routing was not changed).")
    print("- These results measure policy behaviour; they do not establish semantic correctness "
          "without labelled retraction ground truth.")

## 13 — Optional comparison export

Off by default. Flip `EXPORT_COMPARISON` to `True` to write the side-by-side table beside the
experiment artifacts (git-ignored: it carries raw addresses).

In [ ]:
EXPORT_COMPARISON = False

if EXPORT_COMPARISON and ready() and not CONTAMINATED:
    export = master[[RECORD_ID, *SOURCE_FIELDS, "predicted_town", "predicted_country", "town_probability",
                     "country_probability", "EXP01_retracted_address", "EXP02_retracted_address",
                     "exp01_vs_exp02_changed", "comparison_reason"]].rename(columns={
        "EXP01_retracted_address": "exp01_retracted_address", "EXP02_retracted_address": "exp02_retracted_address",
        "exp01_vs_exp02_changed": "changed"})
    export_path = OUTPUT_ROOT / "exp02_vs_exp01_retraction_comparison.csv"
    export.to_csv(export_path, index=False, encoding="utf-8")
    print(f"wrote {len(export)} rows -> {_rel(export_path)}")
else:
    print("Comparison export disabled.")